# 04 – Model Evaluation
Evaluates **persisted** predictions – no models are loaded.

In [1]:
# -------------------------------------------------------------
# Common pre‑amble – read config & export core variables
# -------------------------------------------------------------

from pathlib import Path
import os, sys

# infer repo root from the location of this file
repo_root = os.path.abspath('..')
sys.path.insert(0, str(repo_root))  # allow `import src.*`
from config.notebook_setup import *

Repository Root: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2
Configuration: {'general': {'run_name': 'experiment_with_10_classes', 'seed': 42, 'n_classes': 10}, 'dataset': {'split_type': 'standard'}, 'paths': {'data_exploration_dir': 'output/experiment_with_10_classes/data_exploration', 'artifacts_dir': 'output/experiment_with_10_classes/artifacts', 'embeddings_dir': 'output/experiment_with_10_classes/embeddings', 'models_dir': 'output/experiment_with_10_classes/models', 'results_dir': 'output/experiment_with_10_classes/results'}, 'model': {'embedding_backend': 'sbert', 'sbert_model_name': 'sentence-transformers/all-MiniLM-L12-v2', 'openai_model_name': 'text-embedding-3-small', 'classifier': 'linear_svm', 'rag_top_k': 5, 'use_llm_refine': False}, 'training': {'batch_size': 32, 'max_epochs': 10, 'learning_rate': '1e-3'}, 'evaluation': {'metrics': ['accuracy', 'macro_f1', 'auc_ovr']}}

=== Configuration Variables ===

[DATASET]
  DATASET_SPLIT_TYPE: standard

[EVALUATION]


In [2]:

from src.datasets.dataset import get_dataset
from src.evaluation import run_evaluations

# -------------------------------------------------------------
# 1. Load test labels
# -------------------------------------------------------------

# Load dataset
# X_train, y_train, _, _, _ = get_dataset(split_type="standard", n_classes=N_CLASSES)
X_train, y_train, X_test, y_test, classes = get_dataset(
    split_type=DATASET_SPLIT_TYPE,
    n_classes=N_CLASSES
)

print(f"Loaded {len(X_train)} training documents with {N_CLASSES} classes")

# -------------------------------------------------------------
# 2. Discover model artefacts
# -------------------------------------------------------------
from pathlib import Path
model_names = [p.name for p in Path(MODELS_DIR).iterdir() if (p / "test_predictions.csv").exists()]
print("Found models:", model_names)


INFO | Loading Reuters dataset with configuration:
INFO |   - Split type: standard
INFO |   - Number of classes: 10
INFO |   - Random seed: None
INFO | Loading standard train/test split
INFO | Selected 10 classes: earn, acq, crude, interest, money-fx and more...
INFO | Dataset prepared with:
INFO |   - Training samples: 6337
INFO |   - Test samples: 2477
INFO |   - Classes: 10


Loaded 6337 training documents with 10 classes
Found models: ['RAG-kMajority', 'Naive Bayes', 'RAG-LLM (OpenAI-embeddings)', 'MiniLM + LogReg', 'RAG-CentroidNN', 'RAG-LLM (local-embeddings)', 'TF-IDF bigrams + SVM', 'Linear SVM']


In [ ]:
# -------------------------------------------------------------
# 3. Run evaluation
# -------------------------------------------------------------
results = run_evaluations(
    model_names,
    y_true=y_test,
    y_train_true=y_train,  # Pass training labels
    artefacts_root=MODELS_DIR,
    output_dir=RESULTS_DIR,
    verbose=True,
)


INFO | Starting evaluation process
INFO | Output directory for plots and results: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_10_classes/results
INFO | Output directory for model RAG-kMajority: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_10_classes/results/RAG-kMajority
INFO | Processing distribution analysis for RAG-kMajority on test set
DEBUG | Loaded backend module://matplotlib_inline.backend_inline version unknown.
DEBUG | Loaded backend module://matplotlib_inline.backend_inline version unknown.
DEBUG | findfont: Matching sans\-serif:style=normal:variant=normal:weight=normal:stretch=normal:size=14.399999999999999.
DEBUG | findfont: score(FontEntry(fname='/home/marcmaceira/projects/reuters-rag-classifier_clean_v2/venv/lib/python3.11/site-packages/matplotlib/mpl-data/fonts/ttf/STIXNonUni.ttf', name='STIXNonUnicode', style='normal', variant='normal', weight=400, stretch='normal', size='scalable')) = 10.05
DEBU

In [ ]:
# Display detailed results
import pandas as pd
print("\n=== Summary of Test Metrics ===")
test_metrics = pd.DataFrame({
    model: {k: v for k, v in metrics.items() if k.startswith('test_')}
    for model, metrics in results.items()
}).T
display(test_metrics)

# Show train metrics if available (requires having train predictions saved)
train_cols = [col for col in next(iter(results.values())).keys() if col.startswith('train_')]
if train_cols:
    print("\n=== Summary of Train Metrics ===")
    train_metrics = pd.DataFrame({
        model: {k: v for k, v in metrics.items() if k.startswith('train_')}
        for model, metrics in results.items()
    }).T
    display(train_metrics)
    
    # Show potential overfitting metrics
    diff_cols = [col for col in next(iter(results.values())).keys() if col.endswith('_diff')]
    if diff_cols:
        print("\n=== Train/Test Differences (Overfitting Analysis) ===")
        diff_metrics = pd.DataFrame({
            model: {k: v for k, v in metrics.items() if k.endswith('_diff')}
            for model, metrics in results.items()
        }).T
        display(diff_metrics)
        
        # Interpretation guideline
        print("\nInterpretation guide:")
        print("- Positive values indicate potential overfitting (model performs better on training data)")
        print("- Values close to zero indicate good generalization")
        print("- Negative values might indicate underfitting or data leakage issues")

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

# Print the current results directory to see what's being used
print(f"Results directory: {RESULTS_DIR}")

summary_path = Path(RESULTS_DIR) / "summary_metrics.csv"
print(f"Looking for file at: {summary_path}")

# Check if file exists
if not summary_path.exists():
    print(f"Error: File not found at {summary_path}")
    
    # Look at what's in the results directory
    print(f"\nContents of {RESULTS_DIR}:")
    for item in Path(RESULTS_DIR).iterdir():
        print(f"  - {item.name}")
else:
    # Read the summary metrics
    summary = pd.read_csv(summary_path, index_col=0)
    
    # Function to highlight best values
    def highlight_best(s):
        is_best = s == s.max()
        return ['font-weight: bold' if v else '' for v in is_best]
    
    # Apply styling to the dataframe
    styled_summary = summary.style.apply(highlight_best)
    
    # Display the styled dataframe
    display(styled_summary)